In [ ]:
import json
import os
from typing import List

import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd
import shapely.geometry as sg
from sklearn.cluster import DBSCAN

from notebooks.dataset_utils import read_annotations_folder

RD_EPSG = 28992  # CRS code for the Dutch Rijksdriehoek coordinate system
LAT_LON_EPSG = 4326  # CRS code for WGS84 latitude/longitude coordinate system

MS_PER_RAD: float = 6371008.8  # Earth radius in meters
MIN_SAMPLES: int = 1  # avoid noise points. All points are either in a cluster or are a cluster of their own.


def cluster_and_select_images(gdf: gpd.GeoDataFrame, distance: float = 10.0) -> gpd.GeoDataFrame:
    gdf = gdf.copy()
    points = np.array([[p.x, p.y] for p in gdf["geometry"]])

    # Cluster the points based on distance
    if gdf.crs.to_epsg() == LAT_LON_EPSG:
        epsilon = distance / MS_PER_RAD
        points = np.radians(points)
        metric = "haversine"
    elif gdf.crs.to_epsg() == RD_EPSG:
        epsilon = distance
        metric = "euclidean"
    
    db = DBSCAN(
        eps=epsilon, min_samples=MIN_SAMPLES, algorithm="ball_tree", metric=metric
    ).fit(points)

    gdf["tracking_id"] = db.labels_

    # # For each tracking_id group, discard rows with 'confidence' value below the average of the group
    # conf_select = df["confidence"] >= df.groupby("tracking_id")["confidence"].transform(
    #     "mean"
    # )

    # # Group by cluster and select the image with the largest area
    # df["area"] = df["width"] * df["height"]
    # df["selected"] = False
    # df.iloc[df[conf_select].groupby("tracking_id")["area"].idxmax().values, -1] = True

    return gdf

In [ ]:
dataset_folder = "../datasets/oor/testride_velotech"
coordinates_metadata_file = os.path.join(dataset_folder, "latest_track.json")
annotations_clustered_file = os.path.join(dataset_folder, "clusters_checked_260817.csv")
detections_folder = os.path.join(dataset_folder, "inference/recording_2025-05-14_19-47-40")

categories = {
    2: "Container",
    3: "Dixie",
    4: "Steiger",
}

In [ ]:
with open(coordinates_metadata_file, "r") as f:
    json_content = json.load(f)

data: dict[str, List] = {
    "image_file_name": [],
    "geometry": [],
}

for frame in json_content["frames"]:
    data["image_file_name"].append(frame["image_file_name"])
    data["geometry"].append(
        sg.Point(frame["gps_data"]["longitude"], frame["gps_data"]["latitude"])
    )

coordinates_metadata = (
    gpd.GeoDataFrame(data=data)
    .set_crs(epsg=LAT_LON_EPSG)
    .set_index("image_file_name")
)

In [ ]:
_annotations_clustered = (
    pd.read_csv(annotations_clustered_file, delimiter=";", index_col="id")
)
annotations_clustered = gpd.GeoDataFrame(
    data=_annotations_clustered,
    geometry=_annotations_clustered["image_file_name"].map(coordinates_metadata["geometry"]),
    crs=coordinates_metadata.crs
)

In [ ]:
detections = read_annotations_folder(
    folder_path=detections_folder,
    categories=categories.keys()
)
detections["image_name"] = detections["image_name"] + ".jpg"
detections.rename(columns={"image_name": "image_file_name", "category": "category_id"}, inplace=True)
detections["geometry"] = detections["image_file_name"].map(coordinates_metadata["geometry"])
detections = detections.set_geometry("geometry").set_crs(coordinates_metadata.crs)

In [ ]:
conf = 0.7
_det_conf: gpd.GeoDataFrame = detections[detections["confidence"]>=conf]

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, sharex=True, sharey=True, figsize=(20, 10))

annotations_clustered.to_crs(epsg=RD_EPSG).plot(ax=ax1, column="category_id", cmap="tab20", categorical=True, legend=True)
detections.to_crs(epsg=RD_EPSG).plot(ax=ax2, column="category_id", cmap="tab20", categorical=True, legend=True)
_det_conf.to_crs(epsg=RD_EPSG).plot(ax=ax3, column="category_id", cmap="tab20", categorical=True, legend=True)

ax1.set_title("Manually labelled")
ax2.set_title("Detections @0.1 conf")
ax3.set_title(f"Detections @{conf:.1f} conf")

plt.show()

In [ ]:
cat_id = 2
sample_frac = 0.6

cat_annotations: gpd.GeoDataFrame = annotations_clustered[annotations_clustered["category_id"]==cat_id]
n_clusters = len(cat_annotations["cluster_id"].unique())
print(f"Manual labelling: {n_clusters} clusters for category {cat_id}")

cat_annotations = cluster_and_select_images(
    gdf=cat_annotations, 
    distance=10
)
n_clusters = len(cat_annotations["tracking_id"].unique())
print(f"DBSCAN (manual labelling): {n_clusters} clusters for category {cat_id}")

cat_annotations_sampled = cat_annotations.sample(
    frac=sample_frac,
    axis="index"
)
cat_annotations_sampled = cluster_and_select_images(
    gdf=cat_annotations_sampled, 
    distance=10
)
n_clusters = len(cat_annotations_sampled["tracking_id"].unique())
print(f"DBSCAN (manual labelling, sample {sample_frac:.1f}): {n_clusters} clusters for category {cat_id}")


fig, (ax1, ax2, ax3) = plt.subplots(1, 3, sharex=True, sharey=True, figsize=(20, 10))

cat_annotations.to_crs(epsg=RD_EPSG).plot(ax=ax1, column="cluster_id", cmap="tab20", categorical=True)
cat_annotations.to_crs(epsg=RD_EPSG).plot(ax=ax2, column="tracking_id", cmap="tab20", categorical=True)
cat_annotations_sampled.to_crs(epsg=RD_EPSG).plot(ax=ax3, column="tracking_id", cmap="tab20", categorical=True)

ax1.set_title("Manual clustering")
ax2.set_title("DBSCAN (manual labelling)")
ax3.set_title(f"DBSCAN (manual labelling, sample {sample_frac:.1f})")

plt.show()

In [ ]:
cat_detections: gpd.GeoDataFrame = detections[detections["category_id"]==cat_id]
cat_detections = cluster_and_select_images(
    gdf=cat_detections, 
    distance=20
)
n_clusters = len(cat_detections["tracking_id"].unique())
print(f"Detections @0.1 conf: {n_clusters} clusters for category {cat_id}")

cat_detections_conf: gpd.GeoDataFrame = cat_detections[cat_detections["confidence"]>=conf]
cat_detections_conf = cluster_and_select_images(
    gdf=cat_detections_conf, 
    distance=20
)
n_clusters = len(cat_detections_conf["tracking_id"].unique())
print(f"Detections @{conf:.1f} conf: {n_clusters} clusters for category {cat_id}")


fig, (ax1, ax2, ax3) = plt.subplots(1, 3, sharex=True, sharey=True, figsize=(20, 10))

cat_annotations.to_crs(epsg=RD_EPSG).plot(ax=ax1, column="cluster_id", cmap="tab20", categorical=True)
cat_detections.to_crs(epsg=RD_EPSG).plot(ax=ax2, column="tracking_id", cmap="tab20", categorical=True)
cat_detections_conf.to_crs(epsg=RD_EPSG).plot(ax=ax3, column="tracking_id", cmap="tab20", categorical=True)

ax1.set_title("Manual clustering")
ax2.set_title("DBSCAN (detections @0.1 conf)")
ax3.set_title(f"DBSCAN (detections @{conf:.1f} conf)")

plt.show()